# ETL com Python + IA Generativa

**E**xtract (CSV de usuários) → **T**ransform (IA gera uma mensagem de marketing) → **L**oad (novo CSV).

> Opcional: para usar a IA, adicione o secret `ANTHROPIC_API_KEY` no Colab e ative o acesso ao notebook.

In [1]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.7 MB/s eta 0:00:00


## E — Extract

In [2]:
import pandas as pd

# Dados fictícios (Alternativa 2 do desafio: CSV com Nome, Conta e Cartão)
usuarios = [
    {"Nome": "Ana Souza",     "Conta": "0001-1", "Cartão": "5555 0000 1111 2222"},
    {"Nome": "Bruno Lima",    "Conta": "0002-2", "Cartão": "5555 0000 3333 4444"},
    {"Nome": "Carla Mendes",  "Conta": "0003-3", "Cartão": "5555 0000 5555 6666"},
    {"Nome": "Diego Alves",   "Conta": "0004-4", "Cartão": "5555 0000 7777 8888"},
    {"Nome": "Eduarda Costa", "Conta": "0005-5", "Cartão": "5555 0000 9999 0000"},
]
pd.DataFrame(usuarios).to_csv("usuarios.csv", index=False)

# Extração propriamente dita: ler o CSV
df = pd.read_csv("usuarios.csv")
df

,Nome,Conta,Cartão
0,Ana Souza,0001-1,5555 0000 1111 2222
1,Bruno Lima,0002-2,5555 0000 3333 4444
2,Carla Mendes,0003-3,5555 0000 5555 6666
3,Diego Alves,0004-4,5555 0000 7777 8888
4,Eduarda Costa,0005-5,5555 0000 9999 0000


## T — Transform

In [3]:
from anthropic import Anthropic
from google.colab import userdata

try:
    cliente = Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    print("Usando IA (Claude)")
except Exception:
    cliente = None
    print("Sem chave de API: usando mensagem-modelo")


def gerar_mensagem(nome):
    if cliente:
        try:
            resposta = cliente.messages.create(
                model="claude-haiku-5-5",
                max_tokens=150,
                system="Você é um especialista em marketing bancário.",
                messages=[{
                    "role": "user",
                    "content": f"Crie uma mensagem para {nome} sobre a importância dos "
                               "investimentos (máximo 100 caracteres). Responda só com a mensagem.",
                }],
            )
            return resposta.content[0].text.strip().strip('"')
        except Exception as erro:
            print(f"Falha na IA para {nome}: {erro}")
    return f"{nome}, investir hoje é construir o seu amanhã. Comece agora!"


df["Mensagem"] = df["Nome"].apply(gerar_mensagem)
df

Sem chave de API: usando mensagem-modelo


,Nome,Conta,Cartão,Mensagem
0,Ana Souza,0001-1,5555 0000 1111 2222,"Ana Souza, investir hoje é construir o seu ama..."
1,Bruno Lima,0002-2,5555 0000 3333 4444,"Bruno Lima, investir hoje é construir o seu am..."
2,Carla Mendes,0003-3,5555 0000 5555 6666,"Carla Mendes, investir hoje é construir o seu ..."
3,Diego Alves,0004-4,5555 0000 7777 8888,"Diego Alves, investir hoje é construir o seu a..."
4,Eduarda Costa,0005-5,5555 0000 9999 0000,"Eduarda Costa, investir hoje é construir o seu..."


## L — Load

In [4]:
from google.colab import files

df.to_csv("usuarios_com_mensagem.csv", index=False)
print(f"{len(df)} registros salvos em usuarios_com_mensagem.csv")
files.download("usuarios_com_mensagem.csv")

5 registros salvos em usuarios_com_mensagem.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>